# Latent RAG — Factorial Experiment Runner
Compares traditional vs. latent-space RAG across controlled experiments. The core readout is factorial: keep retrieval fixed to compare generators, and keep generation fixed to compare retrievers.

| # | Retriever | Generator | Research question |
|---|---|---|---|
| 1 | BGE (dense) | Qwen (text) | Strong traditional RAG baseline |
| 2 | BGE (dense) | T5Gemma2 (text) | Can the T5Gemma2 decoder answer from ordinary text context? |
| 3 | BGE (dense) | T5Gemma2 (latent) | Does latent injection work when retrieval is strong? |
| 4 | T5Gemma2 (latent) | Qwen (text) | Is T5Gemma2 latent retrieval competitive? |
| 5 | T5Gemma2 (latent) | T5Gemma2 (text) | Same T5 retriever, normal text-generation control |
| 6 | T5Gemma2 (latent) | T5Gemma2 (latent) | Full latent RAG pipeline |


## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [ ]:
from google.colab import drive
import getpass

token = getpass.getpass('GitHub token (repo read access): ')
hf_token = getpass.getpass('HF_TOKEN')
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

os.chdir('/content')

result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

In [ ]:
%cd /content/latent-rag
!git fetch
!git checkout v3
!git pull origin v3
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

## Download some NQ data
Streams `facebook/dpr` NQs + related documents. Adjust the slice `[:100]` to change dataset size.

In [ ]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Example DPR record:

In [ ]:
import pprint
first = {k: v for k, v in sample[0].items() if k != 'negative_ctxs'}
first['positive_ctxs'] = [{'title': c['title'], 'passage_id': c['passage_id']} for c in first['positive_ctxs']]
pprint.pprint(first, width=120)

## Build mini corpus from NQ passages
Collects all `positive_ctxs` + `hard_negative_ctxs` from the sampled questions.

In [ ]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}
for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

## GPU check & HuggingFace login

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from huggingface_hub import login

login(hf_token)

## Build BGE index (for experiments 1 & 2)
Embedding model: `BAAI/bge-base-en-v1.5` — dense retrieval with [CLS]-token pooling.

In [ ]:
BGE_INDEX_DIR = "/content/index_bge"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {BGE_INDEX_DIR} --retriever_type bge

In [ ]:
!ls -lh {BGE_INDEX_DIR}

## Build latent (T5) index (for experiments 3 & 4)
Embedding model: T5Gemma2 (`google/t5gemma-2-270m-270m` by default) — token-level encoder latents stored in safetensors.
Retrieval uses exhaustive MaxSim over the stored token latents, not a FAISS vector index.
Warning: safetensors file is large (~1.8 GB for 10k passages).

In [ ]:
LATENT_INDEX_DIR = "/content/index_latent"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {LATENT_INDEX_DIR} --retriever_type latent

In [ ]:
!ls -lh {LATENT_INDEX_DIR}

## Build eval file
Construct `data/eval.jsonl` from the DPR dev set. `relevant_ids` are document titles for document-level recall.

In [ ]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

## Run factorial experiments
Each cell runs one experiment. T5Gemma2 generation uses a short output cap because this benchmark scores short Natural Questions answers. Latent-generation cells first run a 5-example quality gate. Results are saved to `results/` and can be aggregated after all experiments finish.


### Experiment 1: BGE + Text (traditional baseline)
BGE dense retrieval → prompted text generation with Qwen.

In [ ]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+text \
    --top_k 5 \
    --max_new_tokens 32 \
    --warmup


### Experiment 2: BGE + T5 Text
BGE dense retrieval → normal text prompt → T5Gemma2 decoder. This is the key same-decoder control for latent generation.


In [ ]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+t5text \
    --top_k 5 \
    --max_new_tokens 16 \
    --warmup


### Experiment 3: BGE + Latent
BGE dense retrieval → T5Gemma2 latent generation. Tests latent injection with strong retrieval.


In [ ]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+latent \
    --top_k 5 \
    --max_new_tokens 16 \
    --max_samples 5 \
    --quality_gate \
    --results_dir smoke_results

!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+latent \
    --top_k 5 \
    --max_new_tokens 16 \
    --warmup


### Experiment 4: T5 + Text
T5Gemma2 encoder MaxSim retrieval → Qwen text generation. Isolates retrieval quality of the T5Gemma2 latent index.


In [ ]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+text \
    --top_k 5 \
    --max_new_tokens 32 \
    --warmup


### Experiment 5: T5 + T5 Text
T5Gemma2 encoder MaxSim retrieval → normal text prompt → T5Gemma2 decoder. Separates T5 retrieval quality from latent generation.


In [ ]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+t5text \
    --top_k 5 \
    --max_new_tokens 16 \
    --warmup


### Experiment 6: T5 + Latent
T5Gemma2 encoder MaxSim retrieval + pre-stored safetensors latents → T5Gemma2 latent generation. Full latent RAG pipeline.


In [ ]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+latent \
    --top_k 5 \
    --max_new_tokens 16 \
    --max_samples 5 \
    --quality_gate \
    --results_dir smoke_results

!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+latent \
    --top_k 5 \
    --max_new_tokens 16 \
    --warmup


## Compare results
Collects all result files from `results/` and displays a side-by-side comparison table.


In [ ]:
import json
from pathlib import Path

results_dir = Path("results")
files = sorted(results_dir.glob("results_*.json"), key=lambda p: p.stat().st_mtime)

COLUMNS = [
    "em",
    "f1",
    "recall@5",
    "answer_support@5",
    "em_when_supported",
    "f1_when_supported",
    "visible_special_tokens",
    "hit_max_new_tokens",
    "max_repeated_3gram",
    "latency_p50_ms",
    "latency_p95_ms",
]

rows = []
for result_file in files:
    with result_file.open() as f:
        data = json.load(f)
    mode = data["config"]["mode"]
    metrics = data["metrics"]
    rows.append((mode, metrics))

if not rows:
    print("No result files found in results/")
else:
    header = f"{'Experiment':<14s}" + "".join(f"{c:>22s}" for c in COLUMNS)
    print(header)
    print("-" * len(header))
    for mode, m in rows:
        vals = ""
        for c in COLUMNS:
            value = m.get(c)
            vals += f"{float(value):>22.4f}" if value is not None else f"{'N/A':>22s}"
        print(f"{mode:<14s}{vals}")
    if len(rows) < 6:
        print(f"\n({len(rows)}/6 experiments completed)")


In [ ]:
!ls -lh /content/latent-rag/results
!cp -r /content/latent-rag/results /content/drive/MyDrive/